In [ ]:
import json

import duckdb
import pandas as pd

from ledger.config import load_settings
from ledger.extract import raw_path
from ledger.frames import districts, read_raw

pd.set_option("display.max_columns", 50, "display.width", 200)
s = load_settings()
paths = {name: str(raw_path(s, src)) for name, src in s.sources.items()}
manifest = json.loads((s.raw_dir / "manifest.json").read_text())
{k: (v["version"], v["rows"]) for k, v in manifest.items()}

In [ ]:
con = duckdb.connect()
con.sql(f"SUMMARIZE SELECT * FROM read_csv('{paths['tcl_change']}')").df()

In [ ]:
chg = read_raw(s, "tcl_change")
summ = read_raw(s, "tcl_summary")
viirs = read_raw(s, "viirs_weekly")
names = districts(read_raw(s, "boundaries"))
chg.shape, summ.shape, viirs.shape, names.shape

In [ ]:
for col in ["threshold", "is_primary", "driver", "iucn_cat", "plantation_type"]:
    print(col, chg[col].value_counts(dropna=False).to_dict(), "\n")

In [ ]:
print(chg["adm1"].nunique(), "states with any loss")
print(chg[["adm1", "adm2"]].drop_duplicates().shape[0], "districts with any loss")
print(names["state"].nunique(), "states in GADM;", len(names), "districts in GADM")
names.groupby("state").size().sort_values(ascending=False).head(10)

In [ ]:
total_all = chg["loss_ha"].sum()
by_t = chg.groupby("threshold")["loss_ha"].sum()
print(f"sum over every row:      {total_all:,.0f} ha")
print(f"sum at threshold 30 only: {by_t[30]:,.0f} ha")
print(f"ratio:                    {total_all / by_t[30]:.1f}x")
by_t.to_frame("loss_ha").assign(vs_30=lambda d: d.loss_ha / d.loc[30, "loss_ha"])

In [ ]:
from ledger.profile import duplicate_keys

key = ["adm1", "adm2", "threshold", "year", "is_primary", "driver", "iucn_cat", "plantation_type"]
dups = duplicate_keys(chg, key)
print(len(dups), "duplicate rows")  # expect 0

In [ ]:
area = summ.groupby(["adm1", "adm2", "threshold"])["area_ha"].sum().unstack("threshold")
print((area.max(axis=1) - area.min(axis=1)).abs().max(), "ha: max spread across thresholds")

# and it should roughly match GADM's own area for the district
a30 = summ[summ.threshold == 30].groupby(["adm1", "adm2"])["area_ha"].sum().rename("gfw_area_ha")
cmp = names.set_index(["adm1", "adm2"])[["gadm_area_ha"]].join(a30)
(cmp.gfw_area_ha / cmp.gadm_area_ha).describe()

In [ ]:
keys = chg[["adm1", "adm2"]].drop_duplicates()
orphans = keys.merge(names, on=["adm1", "adm2"], how="left", indicator=True)
orphans = orphans[orphans["_merge"] == "left_only"]
print(len(orphans), "unnamed districts")
orphans.head()

In [ ]:
names[names.duplicated("district", keep=False)].sort_values("district")[["state", "district"]]

In [ ]:
mine = chg[(chg.threshold == 30) & chg.year.between(2001, 2025)]["loss_ha"].sum()
print(f"{mine / 1e6:.2f} Mha")